<a href="https://colab.research.google.com/github/KamiSir/FlyRank-internship-tasks/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

We need to output a prioritized queue of URLs for the content team. The primary action is "Needs Refresh/Rewrite." To build trust, we will translate the model's features into human-readable reason codes (e.g., flagging if the page is old or has thin content) alongside the risk score.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Load data
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url).fillna(0)

# Prep features and target
df['is_decaying'] = (df['trend_pct'] <= -15.0).astype(int)
features = ['search_volume', 'word_count', 'content_age_days']
X = df[features]
y = df['is_decaying']

# Train a quick model to get probability scores
rf = RandomForestClassifier(max_depth=3, random_state=42)
rf.fit(X, y)

# Score the dataset
df['decay_risk_score'] = rf.predict_proba(X)[:, 1]

# Create human-readable reason codes
def get_reason(row):
    reasons = []
    if row['content_age_days'] > 300: reasons.append("Aging content")
    if row['word_count'] < 1500: reasons.append("Thin word count")
    return " + ".join(reasons) if reasons else "Historical traffic drop pattern"

df['reason_code'] = df.apply(get_reason, axis=1)

# Filter and rank the action queue
action_queue = df[df['decay_risk_score'] > 0.6].sort_values(by='decay_risk_score', ascending=False)

print("--- Top 5 Pages to Refresh This Week ---")
display(action_queue[['content_id', 'decay_risk_score', 'reason_code']].head(5))

--- Top 5 Pages to Refresh This Week ---


,content_id,decay_risk_score,reason_code
10765,content_b8a57a034e50,0.698732,Historical traffic drop pattern
1865,content_b802794c7063,0.698732,Historical traffic drop pattern
17155,content_5515c66c1ea1,0.696974,Historical traffic drop pattern
22127,content_b7da04714513,0.696974,Historical traffic drop pattern
20384,content_594ba0cae917,0.694568,Historical traffic drop pattern


## 2. Intended use and limits

Intended Use: This playbook is designed for SEO and Content Managers to prioritize their weekly editorial calendar and allocate writer resources effectively.

Limits: It is only valid for established content. It stops being valid for brand-new pages (under 90 days old) because they lack the historical baseline needed to measure a traffic drop. It also does not account for sudden, viral traffic spikes that naturally return to baseline.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No code needed for this conceptual section.

## 3. Human review + the no-go list

Human Review Required: An editor must manually review the flagged page to ensure the topic is still relevant to the business strategy before assigning a writer to spend hours rewriting it.

The No-Go List: Do not automatically overwrite, redirect, or delete content based purely on this model's score. Do not apply this prioritization to legal, privacy, or compliance pages, which should only be updated based on regulatory changes.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No code needed for this conceptual section.

## 4. Monitoring / retrain triggers

The model should be monitored on a monthly cadence. Retrain triggers include:

A major Google Core Algorithm Update is confirmed, which alters baseline traffic patterns across the site.

The precision of the top 50 flagged pages drops below 60% (meaning editors are consistently rejecting the model's recommendations).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No code needed for this conceptual section.

## 5. Exports for the paper

Exporting the top 100 prioritized URLs to our outputs folder. We will use this CSV in the final capstone paper to demonstrate a tangible business deliverable.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Ensure directory exists
os.makedirs('work/outputs', exist_ok=True)

# Save the top 100 actions to CSV
export_path = 'work/outputs/action_queue.csv'
action_queue[['content_id', 'decay_risk_score', 'reason_code']].head(100).to_csv(export_path, index=False)

print(f"Successfully exported {len(action_queue.head(100))} prioritized pages to {export_path}")

Successfully exported 100 prioritized pages to work/outputs/action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.